# SMAP soil moisture (NASA Earthdata)

Download a daily SMAP enhanced L3 radiometer soil-moisture granule from the NSIDC DAAC and map the morning (AM) retrieval. HDF5 **raster** on the EASE-2 grid. Live query — needs the `[earthdata]` extra and EDL credentials; wrapped for nbval-lax safety offline.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
paths = None
try:
    paths = EarthLens(
        data_source='earthdata',
        dataset='SPL3SMP_E_006',
        variables=['soil_moisture'],
        start='2023-06-01',
        end='2023-06-01',
        aoi=[-180.0, -90.0, 180.0, 90.0],
        path=str(OUT_DIR),
    ).download(progress_bar=False)
    print(len(paths), 'granule(s):', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

In [ ]:
if paths:
    try:
        import h5py
        import matplotlib.pyplot as plt
        import numpy as np

        with h5py.File(paths[0], 'r') as h5:
            sm = h5['Soil_Moisture_Retrieval_Data_AM/soil_moisture'][:]
        sm = np.where(sm < 0, np.nan, sm)
        fig, ax = plt.subplots(figsize=(10, 5))
        im = ax.imshow(sm, cmap='YlGnBu', vmin=0, vmax=0.5)
        fig.colorbar(im, ax=ax, label='Soil moisture (cm3/cm3)')
        ax.set_title('SMAP enhanced L3 soil moisture (AM), 2023-06-01')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'read/plot step skipped: {type(exc).__name__}: {exc}')